处理notmatched里面的apklist表expanded_apk_policy_url_with_apkid.csv，把这份表里的 policy_link 逐条验活，生成一个清洗版 CSV，新增这些列：

url_status
final_url
is_accessible
notes

同时会尽量保留原表所有字段。

In [5]:
import pandas as pd
import requests
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

In [6]:
INPUT = "LLM_experimental_annotations/expanded_apk_policy_url_with_apkid.csv"
OUTPUT = "LLM_experimental_annotations/expanded_apk_policy_url_with_apkid_validated.csv"

df = pd.read_csv(INPUT)
df.head()

,apkname,policy_name,policy_link,apkid,apkid_confidence,ccpa_relevant_text_or_paraphrase,match_strength,source_file
0,1Password,1Password Privacy Policy,https://1password.com/legal/privacy,com.onepassword.android,high,说明会收集账户、计费、支持、设备和服务使用数据；同时强调保险库内容的加密隔离，体现最小必要收...,High,saudi_pdpl_privacy_policies_50_P10.csv
1,A101 Extra,A101 Extra Kişisel Verilerin Korunması,https://www.a101extra.com/sozlesmeler/kisisel-...,com.a101kapida,high,Privacy/KVKK text for A101 Extra explains reci...,medium,turkey_kvkk_app_privacy_policies_50_R5.csv
2,ACTECON app/service,Privacy Policy & Terms of Use,https://www.actecon.com/en/privacy-policy-term...,NaN,NaN,States users may request to learn whether or n...,strong,kvkk_50_privacy_policies_processed_or_not_R2.csv
3,AJet,AJet Aydınlatma Metni,https://ajet.com/tr/kurumsal/aydinlatma-metni,com.ajet,high,Aydınlatma metni covers ticket sales and mobil...,strong,turkey_kvkk_app_privacy_policies_50_R5.csv
4,Absher Business,Absher Privacy Notice,https://www.absher.sa/wps/vanityurl/en/privacy...,sa.gov.moi.ebusiness,high,Absher publishes a formal privacy notice under...,medium,saudi_pdpl_privacy_policies_50_P13.csv


In [7]:
url_col = "policy_link"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/146.0.0.0 Safari/537.36"
    )
}

In [8]:
def check_url(url: str):
    if pd.isna(url) or not str(url).strip():
        return {
            "url_status": "",
            "final_url": "",
            "is_accessible": False,
            "notes": "empty url",
        }

    url = str(url).strip()
    session = requests.Session()
    session.headers.update(HEADERS)

    try:
        # GET 比 HEAD 更稳，很多站会拦 HEAD
        r = session.get(url, allow_redirects=True, timeout=15)
        final_url = r.url
        status = r.status_code
        ctype = r.headers.get("Content-Type", "")
        text_sample = ""
        if "html" in ctype.lower() or "text" in ctype.lower():
            text_sample = r.text[:800].lower()

        notes = []
        accessible = 200 <= status < 400

        if 300 <= status < 400:
            notes.append("redirect")
        if status >= 400:
            notes.append("http_error")
        if "captcha" in text_sample or "access denied" in text_sample or "forbidden" in text_sample:
            notes.append("possible_blocking")
        if accessible and not notes:
            notes.append("ok")

        return {
            "url_status": status,
            "final_url": final_url,
            "is_accessible": accessible,
            "notes": "; ".join(notes),
        }

    except requests.exceptions.SSLError:
        return {
            "url_status": "ssl_error",
            "final_url": "",
            "is_accessible": False,
            "notes": "ssl error",
        }
    except requests.exceptions.TooManyRedirects:
        return {
            "url_status": "too_many_redirects",
            "final_url": "",
            "is_accessible": False,
            "notes": "too many redirects",
        }
    except requests.exceptions.Timeout:
        return {
            "url_status": "timeout",
            "final_url": "",
            "is_accessible": False,
            "notes": "timeout",
        }
    except requests.exceptions.RequestException as e:
        return {
            "url_status": "request_error",
            "final_url": "",
            "is_accessible": False,
            "notes": str(e)[:200],
        }

unique_urls = df[url_col].dropna().astype(str).str.strip().unique().tolist()
results = {}

with ThreadPoolExecutor(max_workers=16) as ex:
    future_map = {ex.submit(check_url, u): u for u in unique_urls}
    for i, fut in enumerate(as_completed(future_map), 1):
        u = future_map[fut]
        try:
            results[u] = fut.result()
        except Exception as e:
            results[u] = {
                "url_status": "internal_error",
                "final_url": "",
                "is_accessible": False,
                "notes": str(e)[:200],
            }
        if i % 50 == 0:
            print(f"checked {i}/{len(unique_urls)}")

df["url_status"] = df[url_col].astype(str).str.strip().map(lambda u: results.get(u, {}).get("url_status", ""))
df["final_url"] = df[url_col].astype(str).str.strip().map(lambda u: results.get(u, {}).get("final_url", ""))
df["is_accessible"] = df[url_col].astype(str).str.strip().map(lambda u: results.get(u, {}).get("is_accessible", False))
df["notes"] = df[url_col].astype(str).str.strip().map(lambda u: results.get(u, {}).get("notes", ""))
df["url_domain"] = df[url_col].astype(str).str.strip().map(
    lambda u: urlparse(u).netloc if u and u != "nan" else ""
)

df.to_csv(OUTPUT, index=False, encoding="utf-8-sig")
print(f"saved: {OUTPUT}")
print(df["is_accessible"].value_counts(dropna=False))

checked 50/429
checked 100/429
checked 150/429
checked 200/429
checked 250/429
checked 300/429
checked 350/429
checked 400/429
saved: LLM_experimental_annotations/expanded_apk_policy_url_with_apkid_validated.csv
is_accessible
True     521
False    129
Name: count, dtype: int64


从LLM_experimental_annotations/expanded_apk_policy_url_with_apkid_validated.csv提取pp url为true的行所在的行，然后根据privacy policy url去重，之后遍历每一个apk的privacy policy url，然后下载为markdown，以apkname命名,每一个文件保存到目录out_markdown下

In [2]:
import os
import re
import time
import pandas as pd
import requests

d:\softwall_install\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [3]:
# ====== 路径配置 ======
CSV_PATH = "LLM_experimental_annotations/expanded_apk_policy_url_with_apkid_validated.csv"
OUT_DIR = "LLM_experimental_annotations/out_markdown"
os.makedirs(OUT_DIR, exist_ok=True)

In [4]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/146.0.0.0 Safari/537.36"
    )
}

def is_true(v):
    if pd.isna(v):
        return False
    if isinstance(v, bool):
        return v
    s = str(v).strip().lower()
    return s in {"true", "1", "yes", "y", "t"}

def to_bool(v):
    if pd.isna(v):
        return False
    return str(v).strip().lower() in {"true", "1", "yes", "y", "t"}

def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def safe_filename(name, fallback):
    name = str(name or "").strip()
    if not name:
        name = fallback
    name = re.sub(r'[\\/:*?"<>|]', "_", name)  # Windows 文件名非法字符
    name = re.sub(r"\s+", " ", name).strip()
    return name[:120] if name else fallback


In [5]:
df = pd.read_csv(CSV_PATH)
df.head(), df.shape

(               apkname                             policy_name  \
 0            1Password                1Password Privacy Policy   
 1           A101 Extra  A101 Extra Kişisel Verilerin Korunması   
 2  ACTECON app/service           Privacy Policy & Terms of Use   
 3                 AJet                   AJet Aydınlatma Metni   
 4      Absher Business                   Absher Privacy Notice   
 
                                          policy_link                    apkid  \
 0                https://1password.com/legal/privacy  com.onepassword.android   
 1  https://www.a101extra.com/sozlesmeler/kisisel-...           com.a101kapida   
 2  https://www.actecon.com/en/privacy-policy-term...                      NaN   
 3      https://ajet.com/tr/kurumsal/aydinlatma-metni                 com.ajet   
 4  https://www.absher.sa/wps/vanityurl/en/privacy...     sa.gov.moi.ebusiness   
 
   apkid_confidence                   ccpa_relevant_text_or_paraphrase  \
 0             high  说明会收集账户

In [7]:
# 1) 筛选 is_accessible == true 的行
rows = df[df["is_accessible"].map(is_true)].copy()
rows.head(), rows.shape

(                  apkname                    policy_name  \
 0               1Password       1Password Privacy Policy   
 2     ACTECON app/service  Privacy Policy & Terms of Use   
 6   Acronis mobile app(s)      Acronis Privacy Statement   
 7          ActionCard app      ActionCard Privacy Policy   
 8  Adaptavist app/service         Turkish Privacy Policy   
 
                                          policy_link                    apkid  \
 0                https://1password.com/legal/privacy  com.onepassword.android   
 2  https://www.actecon.com/en/privacy-policy-term...                      NaN   
 6        https://www.acronis.com/en/company/privacy/                      NaN   
 7                 https://actioncardapp.com/privacy/                      NaN   
 8  https://www.adaptavist.com/tr-tr/gizlilik-poli...                      NaN   
 
   apkid_confidence                   ccpa_relevant_text_or_paraphrase  \
 0             high  说明会收集账户、计费、支持、设备和服务使用数据；同时强调保险库内容的加密隔离，体现最小

In [9]:
# # 2) policy_link 清洗并去空
# rows["policy_link"] = rows["policy_link"].astype(str).str.strip()
# rows = rows[rows["policy_link"].ne("") & rows["policy_link"].ne("nan")].copy()
# rows.head(), rows.shape

In [11]:
# 3) 按 policy_link 去重（保留第一条）
rows = rows.drop_duplicates(subset=["policy_link"], keep="first").copy()
rows.head(), rows.shape

(                  apkname                    policy_name  \
 0               1Password       1Password Privacy Policy   
 2     ACTECON app/service  Privacy Policy & Terms of Use   
 6   Acronis mobile app(s)      Acronis Privacy Statement   
 7          ActionCard app      ActionCard Privacy Policy   
 8  Adaptavist app/service         Turkish Privacy Policy   
 
                                          policy_link                    apkid  \
 0                https://1password.com/legal/privacy  com.onepassword.android   
 2  https://www.actecon.com/en/privacy-policy-term...                      NaN   
 6        https://www.acronis.com/en/company/privacy/                      NaN   
 7                 https://actioncardapp.com/privacy/                      NaN   
 8  https://www.adaptavist.com/tr-tr/gizlilik-poli...                      NaN   
 
   apkid_confidence                   ccpa_relevant_text_or_paraphrase  \
 0             high  说明会收集账户、计费、支持、设备和服务使用数据；同时强调保险库内容的加密隔离，体现最小

In [12]:
print(f"is_accessible=true 行数（去重前）: {df['is_accessible'].map(is_true).sum()}")
print(f"按 policy_link 去重后: {len(rows)}")

is_accessible=true 行数（去重前）: 521
按 policy_link 去重后: 351


In [13]:
# html -> markdown
try:
    import html2text
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "html2text"])
    import html2text

converter = html2text.HTML2Text()
converter.ignore_links = False
converter.ignore_images = False
converter.body_width = 0

saved = 0
used_names = set()

# 4) 遍历去重后的每个 apk + policy_link，下载并保存 markdown
for idx, row in rows.iterrows():
    url = row["policy_link"]
    apkname = safe_filename(row.get("apkname", ""), fallback=f"apk_{idx}")

    filename = f"{apkname}.md"
    n = 2
    while filename in used_names or os.path.exists(os.path.join(OUT_DIR, filename)):
        filename = f"{apkname}__{n}.md"
        n += 1

    out_path = os.path.join(OUT_DIR, filename)

    try:
        r = requests.get(url, headers=HEADERS, timeout=25, allow_redirects=True)
        if r.status_code >= 400:
            print(f"[SKIP] HTTP {r.status_code} | {url}")
            continue

        md_text = converter.handle(r.text)
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(md_text)

        used_names.add(filename)
        saved += 1
        print(f"[OK] {filename}")
        time.sleep(0.2)
    except Exception as e:
        print(f"[ERR] {apkname} | {url} | {e}")

print(f"完成：共保存 {saved} 个 markdown 文件到 {OUT_DIR}")

[OK] 1Password.md
[OK] ACTECON app_service.md
[OK] Acronis mobile app(s).md
[OK] ActionCard app.md
[OK] Adaptavist app_service.md
[OK] Advantech mobile app(s).md
[OK] Affirm.md
[OK] Agoda.md
[OK] Airbnb.md
[OK] Airbnb__2.md
[OK] Airbnb__3.md
[OK] Airbnb__4.md
[OK] Akbank.md
[OK] Akbank__2.md
[OK] Almosafer.md
[SKIP] HTTP 403 | https://www.amazon.com/gp/help/customer/display.html?nodeId=GX7NJQ4ZB8MHFRNJ
[OK] American Dream app_site.md
[OK] AntiScam app.md
[OK] Anya.md
[OK] Apple Music (Android).md
[OK] Applied Informatics application.md
[OK] Aras Burası.md
[OK] Aras Kargo.md
[OK] Asana.md
[OK] Asana__2.md
[OK] Ascıoğlu mobile app_service.md
[OK] Attio Android app.md
[OK] Auto Tire Car Care mobile app.md
[OK] Avast Mobile Security.md
[OK] Babbel.md
[OK] Base App.md
[OK] Bethany app.md
[OK] BiP.md
[OK] BiTaksi.md
[OK] BiTaksi__2.md
[OK] BiTaksi__3.md
[OK] BiTaksi Sürücü.md
[OK] Biletall.md
[OK] Bimeta app.md
[OK] Binance.md
[OK] Binance__2.md
[OK] Binance.US.md
[OK] Bing.md
[OK] Bitbucket

对这个目录LLM_experimental_annotations\out_markdown下载的md进行清洗:
首先删除内容少于10的md，
然后删除内容是乱码的md，
然后检测到不是英文的md，
最终保存上面的结果

In [14]:
from pathlib import Path
import shutil
import csv
import re
import unicodedata

In [15]:
# =========================
# 配置
# =========================
ROOT_DIR = Path(r"LLM_experimental_annotations\out_markdown")

# 是否真的删除文件：
# False -> 只移动到 removed 子目录，更安全
# True  -> 直接删除
DELETE_DIRECTLY = False

# 少于 10 的判定，这里按“单词数”来做
MIN_WORDS = 10

# 输出日志
LOG_CSV = ROOT_DIR / "cleaning_log.csv"
SUMMARY_TXT = ROOT_DIR / "cleaning_summary.txt"

# 被移走文件的目录
REMOVED_DIR = ROOT_DIR / "_removed"
REMOVED_TOO_SHORT = REMOVED_DIR / "too_short"
REMOVED_GARBLED = REMOVED_DIR / "garbled"
REMOVED_NON_ENGLISH = REMOVED_DIR / "non_english"

In [16]:
# =========================
# 基础工具
# =========================
def read_text_safe(path: Path):
    """
    尝试多种编码读取文本。
    返回: (text, encoding_used, error_msg)
    """
    encodings = ["utf-8", "utf-8-sig", "gb18030", "latin-1"]
    last_error = None

    for enc in encodings:
        try:
            text = path.read_text(encoding=enc, errors="strict")
            return text, enc, ""
        except Exception as e:
            last_error = str(e)

    # 最后兜底，尽量读出来
    try:
        text = path.read_text(encoding="utf-8", errors="replace")
        return text, "utf-8-replace", f"strict decode failed, fallback used: {last_error}"
    except Exception as e:
        return "", "", f"all decode failed: {e}"


def strip_markdown_noise(text: str) -> str:
    """
    粗略去掉 markdown 中对统计干扰较大的符号。
    这里只做轻量清洗，不做完整 markdown 解析。
    """
    t = text

    # 去代码块围栏
    t = re.sub(r"```.*?```", " ", t, flags=re.DOTALL)
    t = re.sub(r"`[^`\n]*`", " ", t)

    # 图片 / 链接
    t = re.sub(r"!\[.*?\]\(.*?\)", " ", t)
    t = re.sub(r"\[(.*?)\]\(.*?\)", r"\1", t)

    # 标题、引用、列表符号
    t = re.sub(r"^[>\-\*\+#\s]+", "", t, flags=re.MULTILINE)

    # 表格竖线替换为空格
    t = t.replace("|", " ")

    # 多空白压缩
    t = re.sub(r"\s+", " ", t).strip()
    return t


def count_words(text: str) -> int:
    """
    统计英文风格单词数。
    """
    words = re.findall(r"\b[a-zA-Z]+(?:'[a-zA-Z]+)?\b", text)
    return len(words)


def printable_ratio(text: str) -> float:
    if not text:
        return 0.0
    printable = sum(ch.isprintable() or ch in "\n\r\t" for ch in text)
    return printable / len(text)


def replacement_char_ratio(text: str) -> float:
    if not text:
        return 0.0
    bad = text.count("�") + text.count("\x00")
    return bad / len(text)


def weird_char_ratio(text: str) -> float:
    """
    统计异常字符比例：
    - 控制字符（保留 \n \r \t）
    - Unicode 分类为 C* 的字符
    """
    if not text:
        return 0.0

    weird = 0
    for ch in text:
        if ch in "\n\r\t":
            continue
        cat = unicodedata.category(ch)
        if cat.startswith("C"):
            weird += 1
    return weird / len(text)


def english_like_ratio(text: str) -> float:
    """
    估计文本中“像英文单词”的比例。
    """
    tokens = re.findall(r"\b[\w'-]+\b", text)
    if not tokens:
        return 0.0

    english_like = 0
    for tok in tokens:
        if re.fullmatch(r"[A-Za-z]+(?:'[A-Za-z]+)?", tok):
            english_like += 1

    return english_like / len(tokens)


def non_ascii_ratio(text: str) -> float:
    if not text:
        return 0.0
    return sum(ord(ch) > 127 for ch in text) / len(text)


def is_garbled(text: str) -> tuple[bool, str]:
    """
    判断是否疑似乱码。
    返回 (是否乱码, 原因说明)
    """
    if not text.strip():
        return True, "empty_after_read"

    rep_ratio = replacement_char_ratio(text)
    weird_ratio = weird_char_ratio(text)
    pr_ratio = printable_ratio(text)
    eng_ratio = english_like_ratio(text)

    # 常见强乱码信号
    if rep_ratio > 0.01:
        return True, f"replacement_char_ratio={rep_ratio:.4f}"

    if weird_ratio > 0.02:
        return True, f"weird_char_ratio={weird_ratio:.4f}"

    if pr_ratio < 0.85:
        return True, f"printable_ratio={pr_ratio:.4f}"

    # 如果看起来几乎没有正常英文 token，同时又不是明显的其他自然语言，可能是乱抽取
    cleaned = strip_markdown_noise(text)
    if len(cleaned) >= 100:
        token_count = len(re.findall(r"\b[\w'-]+\b", cleaned))
        if token_count >= 20 and eng_ratio < 0.20 and non_ascii_ratio(cleaned) < 0.30:
            return True, f"english_like_ratio_too_low={eng_ratio:.4f}"

    return False, ""


def is_non_english(text: str) -> tuple[bool, str]:
    """
    粗略判断是否不是英文。
    不依赖第三方库，规则偏保守。

    思路：
    1. 提取清洗后的文本
    2. 统计英文字母/英文单词占比
    3. 如果非 ASCII 比例过高，且英文单词比例低，则视为非英文
    """
    cleaned = strip_markdown_noise(text)
    if not cleaned:
        return True, "empty_after_clean"

    words = re.findall(r"\b[\w'-]+\b", cleaned)
    if len(words) < 5:
        # 太短的文本不太好判断语言，这里交给上游 too_short
        return False, "too_short_for_language_detection"

    english_words = re.findall(r"\b[A-Za-z]+(?:'[A-Za-z]+)?\b", cleaned)

    word_ratio = len(english_words) / max(len(words), 1)
    ascii_letter_count = sum(("A" <= ch <= "Z") or ("a" <= ch <= "z") for ch in cleaned)
    letter_count = sum(ch.isalpha() for ch in cleaned)
    ascii_letter_ratio = ascii_letter_count / max(letter_count, 1)
    na_ratio = non_ascii_ratio(cleaned)

    # 经验规则，可按数据再调
    # 主要场景：英文 md 的英文字母占比、英文单词占比应明显较高
    if word_ratio < 0.60 and ascii_letter_ratio < 0.70 and na_ratio > 0.20:
        return True, (
            f"word_ratio={word_ratio:.4f}, "
            f"ascii_letter_ratio={ascii_letter_ratio:.4f}, "
            f"non_ascii_ratio={na_ratio:.4f}"
        )

    return False, (
        f"word_ratio={word_ratio:.4f}, "
        f"ascii_letter_ratio={ascii_letter_ratio:.4f}, "
        f"non_ascii_ratio={na_ratio:.4f}"
    )


def ensure_dirs():
    if not DELETE_DIRECTLY:
        REMOVED_TOO_SHORT.mkdir(parents=True, exist_ok=True)
        REMOVED_GARBLED.mkdir(parents=True, exist_ok=True)
        REMOVED_NON_ENGLISH.mkdir(parents=True, exist_ok=True)


def move_or_delete(path: Path, reason: str):
    if DELETE_DIRECTLY:
        path.unlink(missing_ok=True)
        return "deleted"

    mapping = {
        "too_short": REMOVED_TOO_SHORT,
        "garbled": REMOVED_GARBLED,
        "non_english": REMOVED_NON_ENGLISH,
    }
    target_dir = mapping[reason]
    target_path = target_dir / path.name

    # 同名冲突处理
    if target_path.exists():
        stem = path.stem
        suffix = path.suffix
        i = 1
        while True:
            new_target = target_dir / f"{stem}__dup{i}{suffix}"
            if not new_target.exists():
                target_path = new_target
                break
            i += 1

    shutil.move(str(path), str(target_path))
    return f"moved_to:{target_path}"


# =========================
# 主流程
# =========================
def clean_markdown_dir(root_dir: Path):
    if not root_dir.exists():
        raise FileNotFoundError(f"目录不存在: {root_dir}")

    ensure_dirs()

    md_files = sorted(root_dir.glob("*.md"))
    # 如果需要递归遍历子目录，把上面改成:
    # md_files = sorted(root_dir.rglob("*.md"))

    logs = []

    summary = {
        "total_files": 0,
        "kept": 0,
        "removed_too_short": 0,
        "removed_garbled": 0,
        "removed_non_english": 0,
        "read_error": 0,
    }

    for path in md_files:
        # 跳过日志文件自己
        if path.name in {LOG_CSV.name, SUMMARY_TXT.name}:
            continue

        summary["total_files"] += 1

        raw_text, encoding_used, read_error = read_text_safe(path)
        cleaned = strip_markdown_noise(raw_text)
        char_count = len(cleaned)
        word_count = count_words(cleaned)

        record = {
            "file_name": path.name,
            "file_path": str(path),
            "status": "kept",
            "reason": "ok",
            "action": "",
            "encoding": encoding_used,
            "char_count": char_count,
            "word_count": word_count,
            "details": "",
            "read_error": read_error,
        }

        # 1) 内容少于10
        if word_count < MIN_WORDS:
            action = move_or_delete(path, "too_short")
            record["status"] = "removed"
            record["reason"] = "too_short"
            record["action"] = action
            record["details"] = f"word_count={word_count} < {MIN_WORDS}"
            summary["removed_too_short"] += 1
            logs.append(record)
            continue

        # 2) 乱码检测
        garbled, garbled_reason = is_garbled(raw_text)
        if garbled:
            action = move_or_delete(path, "garbled")
            record["status"] = "removed"
            record["reason"] = "garbled"
            record["action"] = action
            record["details"] = garbled_reason
            summary["removed_garbled"] += 1
            logs.append(record)
            continue

        # 3) 非英文检测
        non_en, lang_reason = is_non_english(raw_text)
        if non_en:
            action = move_or_delete(path, "non_english")
            record["status"] = "removed"
            record["reason"] = "non_english"
            record["action"] = action
            record["details"] = lang_reason
            summary["removed_non_english"] += 1
            logs.append(record)
            continue

        # 保留
        record["action"] = "kept"
        record["details"] = lang_reason if 'lang_reason' in locals() else ""
        summary["kept"] += 1
        logs.append(record)

    # 写日志 CSV
    with LOG_CSV.open("w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "file_name",
                "file_path",
                "status",
                "reason",
                "action",
                "encoding",
                "char_count",
                "word_count",
                "details",
                "read_error",
            ],
        )
        writer.writeheader()
        writer.writerows(logs)

    # 写 summary
    with SUMMARY_TXT.open("w", encoding="utf-8") as f:
        for k, v in summary.items():
            f.write(f"{k}: {v}\n")

    print("清洗完成")
    print(f"日志: {LOG_CSV}")
    print(f"汇总: {SUMMARY_TXT}")
    print(summary)

In [17]:
if __name__ == "__main__":
    clean_markdown_dir(ROOT_DIR)

清洗完成
日志: LLM_experimental_annotations\out_markdown\cleaning_log.csv
汇总: LLM_experimental_annotations\out_markdown\cleaning_summary.txt
{'total_files': 350, 'kept': 316, 'removed_too_short': 23, 'removed_garbled': 10, 'removed_non_english': 1, 'read_error': 0}


遍历这个目录LLM_experimental_annotations\out_markdown下一级子目录的markdown文本，不用递归遍历子目录里面的文件，整理一个拆分思路，如何拆才能将很长的md恰好划分为几个部分，然后这几个部分最长1000个单词呢？给个思路，先不给代码.看完md，感觉不需要拆分。